In [ ]:
import os, re, math, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.model_selection import KFold, train_test_split, ParameterGrid
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    balanced_accuracy_score, classification_report
)

In [ ]:
# === Rutas
TR_EMB = Path("/content/drive/MyDrive/TP1/data/deriv/embeddings/train_embeddings_v3.npz")
TE_EMB = Path("/content/drive/MyDrive/TP1/data/deriv/v3/embeddings/test_embeddings_v3.npz")

TRAIN_CSV = Path("/content/drive/MyDrive/TP1/data/deriv/v3/train_unified_aug.csv")
TEST_CSV  = Path("/content/drive/MyDrive/TP1/data/deriv/splits/test_unified.csv")

TARGET_NAME = "valence"
positive_lang = "es"

In [ ]:
_base_aug_pat = re.compile(r"_aug\d+_.*$")
_norm_pat     = re.compile(r"_norm$")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# funciones

In [ ]:
def aggregate_clip(paths, preds, zrms=None, trim=0.10, alpha=0.0, eps=1e-6):
    """
    Agrega predicciones de ventanas a nivel clip con recorte 'trim'
    y ponderación por (zrms^alpha). Si alpha=0, ignora pesos.
    """
    paths = np.asarray(paths)
    preds = np.asarray(preds, dtype=np.float32)

    if zrms is None:
        z = np.ones_like(preds, dtype=np.float32)
    else:
        z = np.asarray(zrms, dtype=np.float32).reshape(-1)
        z = np.maximum(z, 0.0)  # no negativos

    groups = {}
    for i, (p, yhat) in enumerate(zip(paths, preds)):
        g = clip_group_from_path(p)
        lst = groups.setdefault(g, {"p": [], "w": []})
        lst["p"].append(float(yhat))
        lst["w"].append(float(z[i]))

    yhat_clip = {}
    for g, d in groups.items():
        p = np.asarray(d["p"], dtype=np.float64)
        w = np.asarray(d["w"], dtype=np.float64)

        # pesos por alpha (si alpha=0 → todos 1)
        if alpha != 0.0:
            w = np.power(w + eps, float(alpha))
        else:
            w = np.ones_like(w, dtype=np.float64)

        # recorte por predicción
        order = np.argsort(p)
        k = int(np.floor(trim * len(p)))
        if k > 0 and (len(p) - 2 * k) >= 1:
            p = p[order][k:-k]
            w = w[order][k:-k]

        s = w.sum()
        yhat_clip[g] = float(p.mean() if s <= 0 else np.dot(p, w) / s)

    return yhat_clip

In [ ]:
def clip_group_from_path(p: str) -> str:
    s = Path(p).stem
    s = _base_aug_pat.sub("", s)
    s = _norm_pat.sub("", s)
    parent = Path(p).parent.name
    return f"{parent}/{s}"

def ccc(y, yhat, eps=1e-12):
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    mu_y, mu_h = y.mean(), yhat.mean()
    var_y, var_h = y.var(), yhat.var()
    cov = np.mean((y-mu_y)*(yhat-mu_h))
    return float((2*cov)/(var_y + var_h + (mu_y-mu_h)**2 + eps))

def lang_by_clip_from_paths(paths, langs):
    by = {}
    for p, l in zip(paths, langs):
        g = clip_group_from_path(p)
        by.setdefault(g, []).append(l)
    out = {}
    for g, L in by.items():
        u, c = np.unique(np.asarray(L, "U"), return_counts=True)
        out[g] = u[int(np.argmax(c))]
    return out

def lang_weights(langs_fold, target_prior={"es":0.5,"qu":0.5}, cap=4.0, smoothing=1e-3):
    Ls = np.asarray(langs_fold, "U")
    uniq, cnt = np.unique(Ls, return_counts=True)
    L = max(1, len(uniq))
    total = cnt.sum()
    obs = {u:(cnt[i]+smoothing)/(total+smoothing*L) for i,u in enumerate(uniq)}
    ratio = {u: target_prior.get(u, 0.0)/max(obs.get(u,1e-12),1e-12) for u in uniq}
    w = np.array([ratio.get(l,1.0) for l in Ls], np.float32)
    w = np.clip(w, 1.0/cap, cap)
    return w * (len(w)/w.sum())

In [ ]:
def _find_col(df, keys):
    for k in keys:
        if k in df.columns: return k
    raise KeyError(f"No encuentro {keys} en columnas: {df.columns.tolist()}")

In [ ]:
def y_from_paths(paths, m):
    y = np.empty(len(paths), np.float32)
    miss = 0
    for i,p in enumerate(paths):
        if p in m:
            y[i] = float(m[p])
        else:
            # fallback por clip_id base
            g = clip_group_from_path(p)
            # buscar alguna fila con ese clip base
            # (si tu CSV de test no tiene aug, este fallback ayuda)
            hit = df_te[df_te[col_path].apply(lambda s: clip_group_from_path(s)==g)]
            if len(hit):
                y[i] = float(hit.iloc[0][col_val])
            else:
                y[i] = np.nan; miss += 1
    if miss:
        print(f"[Aviso] {miss} paths sin match → NaN")
    return y

In [ ]:
def fit_iso_per_lang(ypc, ytc, langs):
    out = {}
    for lg in np.unique(langs):
        m = (langs==lg)
        if m.sum() < 10:
            out[lg] = None
        else:
            out[lg] = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc[m], ytc[m])
    return out

def apply_iso_per_lang(ypc, langs, cal):
    y = np.empty_like(ypc, float)
    for lg in np.unique(langs):
        m = (langs==lg)
        if cal.get(lg) is None:
            y[m] = ypc[m]
        else:
            y[m] = cal[lg].transform(ypc[m])
    return y

In [ ]:
def _metrics(y, p, tag):
    c = ccc(y, p)
    mae = float(mean_absolute_error(y, p))
    rmse = float(math.sqrt(mean_squared_error(y, p)))
    print(f"=== TEST {tag} ===  MAE={mae:.4f}  RMSE={rmse:.4f}  CCC={c:.4f}")


In [ ]:
def compute_ccc(y_true, y_pred):
    """
    Robust CCC implementation with better numerical stability
    """
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()

    # Remove invalid values
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) < 2:
        return 0.0

    # Pearson correlation
    cor = np.corrcoef(y_true, y_pred)[0, 1]

    # Means and standard deviations
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    std_true = np.std(y_true, ddof=1)
    std_pred = np.std(y_pred, ddof=1)

    # CCC formula using Pearson correlation
    numerator = 2 * cor * std_true * std_pred
    denominator = std_true**2 + std_pred**2 + (mean_true - mean_pred)**2

    if denominator == 0:
        return 0.0

    ccc = numerator / denominator

    return float(ccc)

In [ ]:
from sklearn.model_selection import ParameterGrid
import numpy as np

def optimize_aggregation(yp_val_w, path_val, y_val_w, zrms_val):
    """Optimiza parámetros de agregación en validación"""

    # Grid de búsqueda
    param_grid = {
        'trim': [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20],
        'alpha': [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
    }

    best_ccc = -1
    best_params = None
    results = []

    for params in ParameterGrid(param_grid):
        # Agregar predicciones
        yhat_clip = aggregate_clip(
            path_val, yp_val_w,
            zrms=zrms_val.ravel(),
            trim=params['trim'],
            alpha=params['alpha']
        )

        # Ground truth por clip
        ytrue_clip = {}
        for p, yt in zip(path_val, y_val_w):
            g = clip_group_from_path(p)
            ytrue_clip.setdefault(g, []).append(float(yt))
        ytrue_clip = {g: float(np.mean(v)) for g, v in ytrue_clip.items()}

        # Alinear
        keys = sorted(set(yhat_clip) & set(ytrue_clip))
        ypc = np.array([yhat_clip[k] for k in keys], np.float32)
        ytc = np.array([ytrue_clip[k] for k in keys], np.float32)

        # Calcular CCC
        ccc = compute_ccc(ypc, ytc)
        results.append({'trim': params['trim'], 'alpha': params['alpha'], 'ccc': ccc})

        if ccc > best_ccc:
            best_ccc = ccc
            best_params = params

    print(f"Best aggregation params: trim={best_params['trim']}, alpha={best_params['alpha']}, CCC={best_ccc:.4f}")
    return best_params, results

In [ ]:
def train_xgb(X_train, y_train, sample_weight, X_val=None, y_val=None, params=None):
    params = params or {
        "objective": "reg:squarederror",
        "learning_rate": 0.05,
        "n_estimators": 2000,
        "grow_policy": "lossguide",
        "max_depth": 0,
        "max_leaves": 32,
        "min_child_weight": 10.0,
        "subsample": 0.75,
        "colsample_bytree": 0.80,
        "colsample_bynode": 0.70,
        "gamma": 0.20,
        "reg_alpha": 1.0,
        "reg_lambda": 35.0,
        "max_bin": 128,
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 50,
        "eval_metric": "rmse"
    }

    mdl = XGBRegressor(**params)
    if X_val is not None and y_val is not None:
        mdl.fit(X_train, y_train, sample_weight=sample_weight,
                eval_set=[(X_val, y_val)], verbose=100)
    else:
        mdl.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
    return mdl

In [ ]:
def evaluate_model(mdl, X_tr, y_tr, path_tr, zrms_tr, lang_tr,
                   X_te, y_te, path_te, zrms_te, lang_te,
                   trim=0.10, alpha=0.0):

    # Agregacion (train)
    yp_tr = mdl.predict(X_tr)
    yhat_tr = aggregate_clip(path_tr, yp_tr, zrms_tr, trim, alpha)
    ytrue_tr = {}
    for p, yt in zip(path_tr, y_tr):
        g = clip_group_from_path(p)
        ytrue_tr.setdefault(g, []).append(float(yt))
    ytrue_tr = {g: np.mean(v) for g, v in ytrue_tr.items()}
    keys_tr = sorted(set(yhat_tr) & set(ytrue_tr))
    ypc_tr, ytc_tr = np.array([yhat_tr[k] for k in keys_tr]), np.array([ytrue_tr[k] for k in keys_tr])

    iso_global = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(ypc_tr, ytc_tr)
    langs_tr = np.array([lang_by_clip_from_paths(path_tr, lang_tr)[k] for k in keys_tr])
    cal_lang = fit_iso_per_lang(ypc_tr, ytc_tr, langs_tr)

    # Agregacion (test)
    yp_te = mdl.predict(X_te)
    yhat_te = aggregate_clip(path_te, yp_te, zrms_te, trim, alpha)
    ytrue_te = {}
    for p, yt in zip(path_te, y_te):
        g = clip_group_from_path(p)
        ytrue_te.setdefault(g, []).append(float(yt))
    ytrue_te = {g: np.mean(v) for g, v in ytrue_te.items()}
    keys_te = sorted(set(yhat_te) & set(ytrue_te))
    ypc_te, ytc_te = np.array([yhat_te[k] for k in keys_te]), np.array([ytrue_te[k] for k in keys_te])
    langs_te = np.array([lang_by_clip_from_paths(path_te, lang_te)[k] for k in keys_te])

    # Evalua
    _metrics(ytc_te, ypc_te, "(raw)")
    _metrics(ytc_te, iso_global.transform(ypc_te), "(calibrado GLOBAL)")
    _metrics(ytc_te, apply_iso_per_lang(ypc_te, langs_te, cal_lang), "(calibrado POR IDIOMA)")

    # Clasificacion (3-level)
    ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
    ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_te, cal_lang), bins=[2.5, 3.5], right=True)
    print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
    print(classification_report(ytc_cls, ypc_cls,
          target_names=["Bajo", "Medio", "Alto"], digits=4))

    return iso_global, cal_lang

In [ ]:
def save_model_bundle(model, iso_global, cal_lang, export_path, target_name="valence",
                      positive_lang="es", agg_kwargs=None, xgb_params=None, n_features=None, mu_rms=None, sd_rms=None):
    export_path = Path(export_path)
    export_path.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, export_path / "xgb_model.joblib", compress=3)
    joblib.dump(iso_global, export_path / "iso_global.joblib", compress=3)
    joblib.dump(cal_lang, export_path / "iso_per_lang.joblib", compress=3)

    meta = {
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "target": target_name,
        "positive_lang": positive_lang,
        "agg_kwargs": agg_kwargs or {"trim": 0.10, "alpha": 0.0},
        "xgb_params": xgb_params,
        "n_features": int(n_features or 0),
        "mu_rms": mu_rms,
        "sd_rms": sd_rms
    }
    with open(export_path / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    print("Guardado en:", export_path)

In [ ]:
def concordance_ccc(y_true, y_pred):
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    std_true = np.std(y_true)
    std_pred = np.std(y_pred)
    return (2 * cov) / (std_true**2 + std_pred**2 + (mean_true - mean_pred)**2 + 1e-8)

# carga

In [ ]:
Dt = np.load(TR_EMB, allow_pickle=True)
Xe = Dt["Xw"]

path_tr = Dt["path"].astype("U")
zrms_tr = Dt["zrms"].astype(np.float32)
lang_tr = Dt["lang"].astype("U")
Xw_tr   = Xe.astype(np.float32)

De = np.load(TE_EMB, allow_pickle=True)
Xe = De["Xw"]
path_te = De["path"].astype("U")
zrms_te = De["zrms"].astype(np.float32)
lang_te = De["lang"].astype("U")
Xw_te   = Xe.astype(np.float32)

# columnas extra
is_es_tr = (lang_tr == "es").astype(np.float32)[:, None]
is_es_te = (lang_te == "es").astype(np.float32)[:, None]

Xw_tr = np.concatenate([Xw_tr, is_es_tr, zrms_tr[:, None]], axis=1).astype(np.float32)
Xw_te = np.concatenate([Xw_te, is_es_te, zrms_te[:, None]], axis=1).astype(np.float32)

print("Train windows:", Xw_tr.shape, " Test windows:", Xw_te.shape)


Train windows: (132621, 2050)  Test windows: (15732, 2050)


In [ ]:
WIN_TR_NPZ = Path("/content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_train_v3.npz")

W = np.load(WIN_TR_NPZ, allow_pickle=True)

In [ ]:
mu_rms = float(W["mu_rms"])
sd_rms = float(W["sd_rms"])

In [ ]:
rms_tr = zrms_tr * sd_rms + mu_rms
rms_te = zrms_te * sd_rms + mu_rms

# 2) log1p del RMS
log_tr = np.log1p(np.clip(rms_tr, 1e-9, None))
log_te = np.log1p(np.clip(rms_te, 1e-9, None))

# 3) log con stats del train
mu_log = float(log_tr.mean())
sd_log = float(log_tr.std(ddof=1) + 1e-9)
zrms_log_tr = (log_tr - mu_log) / sd_log
zrms_log_te = (log_te - mu_log) / sd_log

# 4) columna extra
Xw_tr = np.concatenate([Xw_tr, zrms_log_tr[:, None]], axis=1).astype(np.float32)
Xw_te = np.concatenate([Xw_te, zrms_log_te[:, None]], axis=1).astype(np.float32)

print("Nuevas shapes:", Xw_tr.shape, Xw_te.shape)

Nuevas shapes: (132621, 2051) (15732, 2051)


In [ ]:
df_tr = pd.read_csv(TRAIN_CSV)
df_te = pd.read_csv(TEST_CSV)

In [ ]:
df_tr['path_norm'].fillna(df_tr['path'], inplace=True)

In [ ]:
df_tr['is_aug'].fillna(False, inplace=True)

In [ ]:
df_tr['sr'].fillna('', inplace=True)
df_tr['aug_ops'].fillna('', inplace=True)

In [ ]:
col_path = _find_col(df_tr, ["path_norm", "path"])

In [ ]:
col_val  = _find_col(df_tr, ["valence","Valence"])

In [ ]:
map_tr   = df_tr.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
col_path = _find_col(df_te, ["path_norm", "path"])
map_te   = df_te.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
y_tr_w = y_from_paths(path_tr, map_tr)
y_te_w = y_from_paths(path_te, map_te)

In [ ]:
a = dict()
for i in y_tr_w:
  if i>=3.5:
    if 4 not in a:
      a[4] = 1
    else:
      a[4] += 1
  elif i>2.3 and i<3.5:
    if 2 not in a:
      a[2] = 1
    else:
      a[2] += 1
  else:
    if 1 not in a:
      a[1] = 1
    else:
      a[1]+=1

print("valence",a)

valence {1: 33735, 2: 57451, 4: 41435}


In [ ]:
mtr = np.isfinite(y_tr_w); mte = np.isfinite(y_te_w)
Xw_tr, path_tr, zrms_tr, lang_tr, y_tr_w = Xw_tr[mtr], path_tr[mtr], zrms_tr[mtr], lang_tr[mtr], y_tr_w[mtr]
Xw_te, path_te, zrms_te, lang_te, y_te_w = Xw_te[mte], path_te[mte], zrms_te[mte], lang_te[mte], y_te_w[mte]

print("Targets TRAIN (min/mean/max):", float(y_tr_w.min()), float(y_tr_w.mean()), float(y_tr_w.max()))
print("Targets TEST  (min/mean/max):", float(y_te_w.min()), float(y_te_w.mean()), float(y_te_w.max()))

Targets TRAIN (min/mean/max): 1.0 3.0483312606811523 5.0
Targets TEST  (min/mean/max): 1.0 3.102180242538452 5.0


In [ ]:
g_tr = np.array([clip_group_from_path(p) for p in path_tr])

# pesos por clip
ug, cnt = np.unique(g_tr, return_counts=True)
cnt_map = {g:c for g,c in zip(ug, cnt)}
sw_clip = np.array([1.0/cnt_map[g] for g in g_tr], dtype=np.float32)
sw_clip *= (len(sw_clip) / sw_clip.sum())  # media≈1

# 50/50 para ambos idiomas
prior = {"es": 0.5, "qu": 0.5}

# pesos por idioma
w_lang = lang_weights(lang_tr, target_prior=prior)

# combina, clip y renormaliza
W_full = np.clip(sw_clip * w_lang, 1e-3, 20.0).astype(np.float32)
W_full *= (len(W_full) / W_full.sum())  # media≈1

print("W_full: min=%.3f  p50=%.3f  p90=%.3f  max=%.3f  mean=%.3f" % (
    float(W_full.min()), float(np.median(W_full)), float(np.quantile(W_full, 0.90)),
    float(W_full.max()), float(W_full.mean())
))


W_full: min=0.393  p50=0.787  p90=1.574  max=7.869  mean=1.000


# Pruebas

## primero

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.10,
    "n_estimators": 1200,
    "grow_policy": "lossguide",
    "max_depth": 0, "max_leaves": 48,
    "min_child_weight": 6.0,
    "subsample": 0.90,
    "colsample_bytree": 1.0,
    "colsample_bynode": 0.80,
    "gamma": 0.10,
    "reg_alpha": 0.0, "reg_lambda": 24.0,
    "max_bin": 128,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

In [ ]:
mdl = XGBRegressor(**best_params)
mdl.fit(Xw_tr, y_tr_w, sample_weight=W_full, verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.8, colsample_bytree=1.0,
             device=None, early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=0.1, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=128,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=48,
             min_child_weight=6.0, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1200, n_jobs=-1,
             num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te,
    trim=0.10, alpha=0.0
)


=== TRAIN (clip, raw) ===  MAE=0.1970 RMSE=0.2646 CCC=0.9497
=== TEST (raw) ===  MAE=0.4995  RMSE=0.6269  CCC=0.6232
=== TEST (calibrado GLOBAL) ===  MAE=0.4824  RMSE=0.6186  CCC=0.6746
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4680  RMSE=0.6154  CCC=0.6823


lo siguiente se ha incorporado en la nueva funcion "evaluate_model"

In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.5709410164444079
              precision    recall  f1-score   support

        Bajo     0.7696    0.4355    0.5562       721
       Medio     0.5308    0.7629    0.6260      1050
        Alto     0.6614    0.5145    0.5788       657

    accuracy                         0.5984      2428
   macro avg     0.6540    0.5709    0.5870      2428
weighted avg     0.6371    0.5984    0.5925      2428



guardamos modelo prueba 1

In [ ]:
agg_kwargs = {"trim":0.10,"alpha":0.0}

In [ ]:
save_model_bundle(
    mdl, iso_global, cal_lang,
    "/content/drive/MyDrive/TP1/modelos/valencia/v4",
    target_name="valence",
    agg_kwargs
    positive_lang="es",
    xgb_params=best_params,
    n_features=Xw_tr.shape[1]
)

Guardado en: /content/drive/MyDrive/TP1/modelos/valencia/v3


## segundo

mejores resultados con esta configuracion

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "n_estimators": 2000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "max_leaves": 32,
    "min_child_weight": 10.0,
    "subsample": 0.75,
    "colsample_bytree": 0.80,
    "colsample_bynode": 0.70,
    "gamma": 0.20,
    "reg_alpha": 1.0,
    "reg_lambda": 35.0,
    "max_bin": 128,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "early_stopping_rounds": 50,
    "eval_metric": "rmse"
}

In [ ]:
from sklearn.model_selection import train_test_split
(X_train, X_val,
 y_train, y_val,
 W_train, W_val,
 path_train, path_val,
 zrms_train, zrms_val,
 lang_train, lang_val) = train_test_split(
    Xw_tr, y_tr_w, W_full, path_tr, zrms_tr, lang_tr,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Weight stats - Train: min={W_train.min():.3f}, mean={W_train.mean():.3f}, max={W_train.max():.3f}")
print(f"Weight stats - Val: min={W_val.min():.3f}, mean={W_val.mean():.3f}, max={W_val.max():.3f}")

Training samples: 112727
Validation samples: 19894
Weight stats - Train: min=0.393, mean=0.999, max=7.869
Weight stats - Val: min=0.393, mean=1.005, max=7.869


In [ ]:
mdl = train_xgb(
    X_train, y_train, W_train,
    X_val=X_val, y_val=y_val,
    params=best_params
)

[0]	validation_0-rmse:0.91899
[50]	validation_0-rmse:0.75502
[100]	validation_0-rmse:0.71988
[150]	validation_0-rmse:0.69987
[200]	validation_0-rmse:0.68638
[250]	validation_0-rmse:0.67566
[300]	validation_0-rmse:0.66725
[350]	validation_0-rmse:0.66067
[400]	validation_0-rmse:0.65454
[450]	validation_0-rmse:0.64949
[500]	validation_0-rmse:0.64508
[550]	validation_0-rmse:0.64125
[600]	validation_0-rmse:0.63783
[650]	validation_0-rmse:0.63473
[700]	validation_0-rmse:0.63215
[750]	validation_0-rmse:0.62934
[800]	validation_0-rmse:0.62640
[850]	validation_0-rmse:0.62406
[900]	validation_0-rmse:0.62176
[950]	validation_0-rmse:0.61957
[1000]	validation_0-rmse:0.61739
[1050]	validation_0-rmse:0.61534
[1100]	validation_0-rmse:0.61314
[1150]	validation_0-rmse:0.61151
[1200]	validation_0-rmse:0.60987
[1250]	validation_0-rmse:0.60829
[1300]	validation_0-rmse:0.60670
[1350]	validation_0-rmse:0.60529
[1400]	validation_0-rmse:0.60378
[1450]	validation_0-rmse:0.60244
[1500]	validation_0-rmse:0.60105


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.7, colsample_bytree=0.8,
             device=None, early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.2, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=128,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=32,
             min_child_weight=10.0, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=2000, n_jobs=-1,
             num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te
)


=== TRAIN (clip, raw) ===  MAE=0.2956 RMSE=0.3889 CCC=0.8827
=== TEST (raw) ===  MAE=0.5022  RMSE=0.6267  CCC=0.6204
=== TEST (calibrado GLOBAL) ===  MAE=0.4842  RMSE=0.6160  CCC=0.6877
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4725  RMSE=0.6129  CCC=0.6945


lo siguiente se ha incorporado en la nueva funcion "evaluate_model"

In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.5792551497405867
              precision    recall  f1-score   support

        Bajo     0.7585    0.4660    0.5773       721
       Medio     0.5383    0.7162    0.6146      1050
        Alto     0.6207    0.5556    0.5863       657

    accuracy                         0.5984      2428
   macro avg     0.6392    0.5793    0.5928      2428
weighted avg     0.6260    0.5984    0.5959      2428



In [ ]:
agg_kwargs = {"trim":0.10,"alpha":0.0}

In [ ]:
save_model_bundle(
    mdl, iso_global, cal_lang,
    "/content/drive/MyDrive/TP1/modelos/valencia/v3/segundo",
    target_name="valence",
    positive_lang="es",
    xgb_params=best_params,
    n_features=Xw_tr.shape[1]
)


## tercero

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.03,
    "n_estimators": 3000,
    "grow_policy": "depthwise",
    "max_depth": 6,
    "min_child_weight": 15.0,
    "subsample": 0.70,
    "colsample_bytree": 0.75,
    "colsample_bynode": 0.65,
    "gamma": 0.30,
    "reg_alpha": 2.0,
    "reg_lambda": 40.0,
    "max_bin": 96,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

In [ ]:
# Train model
mdl = train_xgb(
    X_train, y_train, W_train,
    X_val=X_val, y_val=y_val,
    params=best_params
)

[0]	validation_0-rmse:0.92295
[50]	validation_0-rmse:0.77862
[100]	validation_0-rmse:0.73615
[150]	validation_0-rmse:0.71552
[200]	validation_0-rmse:0.70131
[250]	validation_0-rmse:0.69044
[300]	validation_0-rmse:0.68153
[350]	validation_0-rmse:0.67424
[400]	validation_0-rmse:0.66796
[450]	validation_0-rmse:0.66289
[500]	validation_0-rmse:0.65807
[550]	validation_0-rmse:0.65377
[600]	validation_0-rmse:0.64967
[650]	validation_0-rmse:0.64620
[700]	validation_0-rmse:0.64277
[750]	validation_0-rmse:0.63935
[800]	validation_0-rmse:0.63655
[850]	validation_0-rmse:0.63373
[900]	validation_0-rmse:0.63112
[950]	validation_0-rmse:0.62871
[1000]	validation_0-rmse:0.62626
[1050]	validation_0-rmse:0.62408
[1100]	validation_0-rmse:0.62193
[1150]	validation_0-rmse:0.61997
[1200]	validation_0-rmse:0.61812
[1250]	validation_0-rmse:0.61632
[1300]	validation_0-rmse:0.61457
[1350]	validation_0-rmse:0.61276
[1400]	validation_0-rmse:0.61111
[1450]	validation_0-rmse:0.60951
[1500]	validation_0-rmse:0.60813


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.65,
             colsample_bytree=0.75, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.3, grow_policy='depthwise',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=96, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=15.0, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=3000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te
)


=== TRAIN (clip, raw) ===  MAE=0.2652 RMSE=0.3551 CCC=0.9040
=== TEST (raw) ===  MAE=0.5025  RMSE=0.6281  CCC=0.6175
=== TEST (calibrado GLOBAL) ===  MAE=0.4844  RMSE=0.6182  CCC=0.6824
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4712  RMSE=0.6142  CCC=0.6899


lo siguiente se ha incorporado en la nueva funcion "evaluate_model"

In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.5796933201322082
              precision    recall  f1-score   support

        Bajo     0.7739    0.4605    0.5774       721
       Medio     0.5373    0.7276    0.6181      1050
        Alto     0.6274    0.5510    0.5867       657

    accuracy                         0.6005      2428
   macro avg     0.6462    0.5797    0.5941      2428
weighted avg     0.6319    0.6005    0.5975      2428



## validacion por folds (ver generalizacion del modelo)

con la mejor configuración

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "n_estimators": 2000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "max_leaves": 32,
    "min_child_weight": 10.0,
    "subsample": 0.75,
    "colsample_bytree": 0.80,
    "colsample_bynode": 0.70,
    "gamma": 0.20,
    "reg_alpha": 1.0,
    "reg_lambda": 35.0,
    "max_bin": 128,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
    "early_stopping_rounds": 50,
    "eval_metric": "rmse"
}

In [ ]:
(X_train, X_val,
 y_train, y_val,
 W_train, W_val,
 path_train, path_val,
 zrms_train, zrms_val,
 lang_train, lang_val) = train_test_split(
    Xw_tr, y_tr_w, W_full, path_tr, zrms_tr, lang_tr,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

In [ ]:
ccc_scores = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(Xw_tr):
    # Train/val splits con arrays
    X_tr, X_val = Xw_tr[train_idx], Xw_tr[val_idx]
    y_tr, y_val = y_tr_w[train_idx], y_tr_w[val_idx]
    w_tr = W_full[train_idx]

    path_tr_fold = path_tr[train_idx]
    path_val_fold = path_tr[val_idx]
    zrms_tr_fold = zrms_tr[train_idx]
    zrms_val_fold = zrms_tr[val_idx]
    lang_tr_fold = lang_tr[train_idx]
    lang_val_fold = lang_tr[val_idx]

    # --- Modelo base ---
    model = XGBRegressor(**best_params)
    model.fit(X_tr, y_tr, sample_weight=w_tr,
              eval_set=[(X_val, y_val)],
              verbose=150)

    # --- Predicciones entrenamiento (para calibrar) ---
    yp_tr = model.predict(X_tr).astype(np.float32)
    yhat_tr_clip = aggregate_clip(path_tr_fold, yp_tr, zrms=zrms_tr_fold.ravel(), trim=0.10, alpha=0.0)

    ytrue_tr_clip = {}
    for p, yt in zip(path_tr_fold, y_tr):
        g = clip_group_from_path(p)
        ytrue_tr_clip.setdefault(g, []).append(float(yt))
    ytrue_tr_clip = {g: float(np.mean(v)) for g, v in ytrue_tr_clip.items()}

    keys_tr = sorted(set(yhat_tr_clip) & set(ytrue_tr_clip))
    ypc_tr = np.array([yhat_tr_clip[k] for k in keys_tr], np.float32)
    ytc_tr = np.array([ytrue_tr_clip[k] for k in keys_tr], np.float32)
    clip2lang_tr = lang_by_clip_from_paths(path_tr_fold, lang_tr_fold)
    langs_clip_tr = np.array([clip2lang_tr[k] for k in keys_tr])

    # --- Calibrador por idioma ---
    calibradores = fit_iso_per_lang(ypc_tr, ytc_tr, langs_clip_tr)

    # --- Predicciones validación ---
    yp_val = model.predict(X_val).astype(np.float32)
    yhat_val_clip = aggregate_clip(path_val_fold, yp_val, zrms=zrms_val_fold.ravel(), trim=0.10, alpha=0.0)

    ytrue_val_clip = {}
    for p, yt in zip(path_val_fold, y_val):
        g = clip_group_from_path(p)
        ytrue_val_clip.setdefault(g, []).append(float(yt))
    ytrue_val_clip = {g: float(np.mean(v)) for g, v in ytrue_val_clip.items()}

    keys_val = sorted(set(yhat_val_clip) & set(ytrue_val_clip))
    ypc_val = np.array([yhat_val_clip[k] for k in keys_val], np.float32)
    ytc_val = np.array([ytrue_val_clip[k] for k in keys_val], np.float32)
    clip2lang_val = lang_by_clip_from_paths(path_val_fold, lang_val_fold)
    langs_clip_val = np.array([clip2lang_val[k] for k in keys_val])

    # --- Calibrar predicciones ---
    ypc_val_cal = np.array([
        calibradores[lang].predict([[yp]])[0] if lang in calibradores else yp
        for yp, lang in zip(ypc_val, langs_clip_val)
    ])

    # --- CCC calibrado ---
    ccc = concordance_ccc(ytc_val, ypc_val_cal)
    ccc_scores.append(ccc)

print(f"\n=== CCC por fold calibrado === Mean= {np.mean(ccc_scores):.4f}, Std= {np.std(ccc_scores):.4f}")

[0]	validation_0-rmse:0.91932
[150]	validation_0-rmse:0.69971
[300]	validation_0-rmse:0.66749
[450]	validation_0-rmse:0.64995
[600]	validation_0-rmse:0.63776
[750]	validation_0-rmse:0.62828
[900]	validation_0-rmse:0.62098
[1050]	validation_0-rmse:0.61425
[1200]	validation_0-rmse:0.60856
[1350]	validation_0-rmse:0.60383
[1500]	validation_0-rmse:0.59972
[1650]	validation_0-rmse:0.59559
[1800]	validation_0-rmse:0.59227
[1950]	validation_0-rmse:0.58920
[1999]	validation_0-rmse:0.58818
[0]	validation_0-rmse:0.91931
[150]	validation_0-rmse:0.69682
[300]	validation_0-rmse:0.66484
[450]	validation_0-rmse:0.64728
[600]	validation_0-rmse:0.63544
[750]	validation_0-rmse:0.62617
[900]	validation_0-rmse:0.61883
[1050]	validation_0-rmse:0.61274
[1200]	validation_0-rmse:0.60746
[1350]	validation_0-rmse:0.60291
[1500]	validation_0-rmse:0.59846
[1650]	validation_0-rmse:0.59471
[1800]	validation_0-rmse:0.59133
[1950]	validation_0-rmse:0.58792
[1999]	validation_0-rmse:0.58679
[0]	validation_0-rmse:0.9192

# Modelo final - valencia

nuevas agregaciones(mfcc_mean, mfcc_std, pitch_features)

In [ ]:
ACOUSTIC_FEAT_DIR = Path("/content/drive/MyDrive/TP1/data/deriv/v3/acoustic_features")
WIN_TR_NPZ = Path("/content/drive/MyDrive/TP1/data/deriv/v3/windows/windows_train_v3.npz")

In [ ]:
Dt = np.load(TR_EMB, allow_pickle=True)
Xw_tr, path_tr, zrms_tr, lang_tr = Dt["Xw"], Dt["path"], Dt["zrms"], Dt["lang"]

De = np.load(TE_EMB, allow_pickle=True)
Xw_te, path_te, zrms_te, lang_te = De["Xw"], De["path"], De["zrms"], De["lang"]

In [ ]:
AF_tr = np.load(ACOUSTIC_FEAT_DIR / "train_acoustic_features_v3.npz")
acoustic_feats_tr = AF_tr["acoustic_features"]
AF_te = np.load(ACOUSTIC_FEAT_DIR / "test_acoustic_features_v3.npz")
acoustic_feats_te = AF_te["acoustic_features"]

# --- 2. Preparación de todas las características ---

# Normaliza las nuevas características acústicas
mu_ac = np.mean(acoustic_feats_tr, axis=0)
sd_ac = np.std(acoustic_feats_tr, axis=0) + 1e-8
acoustic_feats_tr_norm = (acoustic_feats_tr - mu_ac) / sd_ac
acoustic_feats_te_norm = (acoustic_feats_te - mu_ac) / sd_ac

# Prepara la columna 'is_es'
is_es_tr = (lang_tr == "es").astype(np.float32)[:, None]
is_es_te = (lang_te == "es").astype(np.float32)[:, None]

In [ ]:
acoustic_norm = {
    "mu_ac": mu_ac.tolist(),
    "sd_ac": sd_ac.tolist()
}
with open("/content/drive/MyDrive/TP1/modelos/valencia/v3/acoustic_val.json", "w") as f:
    json.dump(acoustic_norm, f, indent=2)

In [ ]:
W = np.load(WIN_TR_NPZ, allow_pickle=True)
mu_rms, sd_rms = float(W["mu_rms"]), float(W["sd_rms"])
rms_tr = zrms_tr * sd_rms + mu_rms
log_tr = np.log1p(np.clip(rms_tr, 1e-9, None))
mu_log, sd_log = float(log_tr.mean()), float(log_tr.std(ddof=1) + 1e-9)
zrms_log_tr = ((log_tr - mu_log) / sd_log)[:, None] # Asegura que sea una columna
rms_te = zrms_te * sd_rms + mu_rms
log_te = np.log1p(np.clip(rms_te, 1e-9, None))
zrms_log_te = ((log_te - mu_log) / sd_log)[:, None] # Asegura que sea una columna

In [ ]:
print("Concatena conjunto de datos")
Xw_tr = np.concatenate([
    Xw_tr.astype(np.float32),      # Embeddings wav2vec
    is_es_tr,                      # Es español?
    zrms_tr[:, None],              # Z-score de RMS
    zrms_log_tr,                   # Z-score del log de RMS
    acoustic_feats_tr_norm         # Pitch y MFCCs
], axis=1)

Xw_te = np.concatenate([
    Xw_te.astype(np.float32),      # Embeddings wav2vec
    is_es_te,                      # Es español?
    zrms_te[:, None],              # Z-score de RMS
    zrms_log_te,                   # Z-score del log de RMS
    acoustic_feats_te_norm         # Pitch y MFCCs
], axis=1)

print(f"Shape final de entrenamiento: {Xw_tr.shape}")
print(f"Shape final de test: {Xw_te.shape}")

In [ ]:
df_tr = pd.read_csv(TRAIN_CSV)
df_te = pd.read_csv(TEST_CSV)

In [ ]:
df_tr['path_norm'].fillna(df_tr['path'], inplace=True)

In [ ]:
df_tr['is_aug'].fillna(False, inplace=True)

In [ ]:
df_tr['sr'].fillna('', inplace=True)
df_tr['aug_ops'].fillna('', inplace=True)

In [ ]:
df_tr

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,path_norm,sr,is_aug,aug_ops
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_10-02-03-04,2.0,3.0,4.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_104-03-02-04,3.0,2.0,4.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_1-02-03-03,2.0,3.0,3.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_105-03-02-04,3.0,2.0,4.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio1,Audio1_102-03-04-05,3.0,4.0,5.0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,,False,
...,...,...,...,...,...,...,...,...,...,...,...,...
20146,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_83-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ts1.013
20147,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ns32.2
20148,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.0,2.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,ns33.2
20149,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_86-03-03-04,3.0,3.0,4.0,/content/drive/MyDrive/TP1/data/deriv/v3/train...,16000.0,True,rvb0.12s_0.11


In [ ]:
col_path = _find_col(df_tr, ["path_norm", "path"])

In [ ]:
col_val  = _find_col(df_tr, ["valence","Valence"])

In [ ]:
map_tr   = df_tr.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
col_path = _find_col(df_te, ["path_norm", "path"])
map_te   = df_te.set_index(col_path)[col_val].astype(float).to_dict()

In [ ]:
y_tr_w = y_from_paths(path_tr, map_tr)
y_te_w = y_from_paths(path_te, map_te)

In [ ]:
mtr = np.isfinite(y_tr_w); mte = np.isfinite(y_te_w)
Xw_tr, path_tr, zrms_tr, lang_tr, y_tr_w = Xw_tr[mtr], path_tr[mtr], zrms_tr[mtr], lang_tr[mtr], y_tr_w[mtr]
Xw_te, path_te, zrms_te, lang_te, y_te_w = Xw_te[mte], path_te[mte], zrms_te[mte], lang_te[mte], y_te_w[mte]

print("Targets TRAIN (min/mean/max):", float(y_tr_w.min()), float(y_tr_w.mean()), float(y_tr_w.max()))
print("Targets TEST  (min/mean/max):", float(y_te_w.min()), float(y_te_w.mean()), float(y_te_w.max()))

Targets TRAIN (min/mean/max): 1.0 2.959321975708008 5.0
Targets TEST  (min/mean/max): 1.0 3.0794599056243896 5.0


In [ ]:
g_tr = np.array([clip_group_from_path(p) for p in path_tr])

# pesos por clip
ug, cnt = np.unique(g_tr, return_counts=True)
cnt_map = {g:c for g,c in zip(ug, cnt)}
sw_clip = np.array([1.0/cnt_map[g] for g in g_tr], dtype=np.float32)
sw_clip *= (len(sw_clip) / sw_clip.sum())  # media≈1

# prior de idioma
prior = {"es": 0.5, "qu": 0.5}

# pesos por idioma
w_lang = lang_weights(lang_tr, target_prior=prior)

# combina, clip y renormaliza
W_full = np.clip(sw_clip * w_lang, 1e-3, 20.0).astype(np.float32)
W_full *= (len(W_full) / W_full.sum())  # media≈1

print("W_full: min=%.3f  p50=%.3f  p90=%.3f  max=%.3f  mean=%.3f" % (
    float(W_full.min()), float(np.median(W_full)), float(np.quantile(W_full, 0.90)),
    float(W_full.max()), float(W_full.mean())
))


W_full: min=0.393  p50=0.787  p90=1.574  max=7.869  mean=1.000


usamos mejor configuracion vista previamente

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "n_estimators": 2000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "max_leaves": 32,
    "min_child_weight": 10.0,
    "subsample": 0.75,
    "colsample_bytree": 0.80,
    "colsample_bynode": 0.70,
    "gamma": 0.20,
    "reg_alpha": 1.0,
    "reg_lambda": 35.0,
    "max_bin": 128,
    "tree_method": "hist",
    "random_state": 42,
    "device":"cuda",
    "n_jobs": -1,
    "early_stopping_rounds": 50,
    "eval_metric": "rmse"
}

In [ ]:
from sklearn.model_selection import train_test_split
(X_train, X_val,
 y_train, y_val,
 W_train, W_val,
 path_train, path_val,
 zrms_train, zrms_val,
 lang_train, lang_val) = train_test_split(
    Xw_tr, y_tr_w, W_full, path_tr, zrms_tr, lang_tr,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Weight stats - Train: min={W_train.min():.3f}, mean={W_train.mean():.3f}, max={W_train.max():.3f}")
print(f"Weight stats - Val: min={W_val.min():.3f}, mean={W_val.mean():.3f}, max={W_val.max():.3f}")

Training samples: 112727
Validation samples: 19894
Weight stats - Train: min=0.393, mean=0.999, max=7.869
Weight stats - Val: min=0.393, mean=1.005, max=7.869


In [ ]:
mdl = train_xgb(
    X_train, y_train, W_train,
    X_val=X_val, y_val=y_val,
    params=best_params
)

[0]	validation_0-rmse:0.91786
[50]	validation_0-rmse:0.71304
[100]	validation_0-rmse:0.67897
[150]	validation_0-rmse:0.66185
[200]	validation_0-rmse:0.64929
[250]	validation_0-rmse:0.64035
[300]	validation_0-rmse:0.63334
[350]	validation_0-rmse:0.62711
[400]	validation_0-rmse:0.62211
[450]	validation_0-rmse:0.61802
[500]	validation_0-rmse:0.61444
[550]	validation_0-rmse:0.61139
[600]	validation_0-rmse:0.60826
[650]	validation_0-rmse:0.60527
[700]	validation_0-rmse:0.60273
[750]	validation_0-rmse:0.60011
[800]	validation_0-rmse:0.59797
[850]	validation_0-rmse:0.59568
[900]	validation_0-rmse:0.59362
[950]	validation_0-rmse:0.59173
[1000]	validation_0-rmse:0.58976
[1050]	validation_0-rmse:0.58800
[1100]	validation_0-rmse:0.58644
[1150]	validation_0-rmse:0.58477
[1200]	validation_0-rmse:0.58307
[1250]	validation_0-rmse:0.58136
[1300]	validation_0-rmse:0.57994
[1350]	validation_0-rmse:0.57888
[1400]	validation_0-rmse:0.57787
[1450]	validation_0-rmse:0.57663
[1500]	validation_0-rmse:0.57554


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=0.7, colsample_bytree=0.8,
             device='cuda', early_stopping_rounds=50, enable_categorical=False,
             eval_metric='rmse', feature_types=None, feature_weights=None,
             gamma=0.2, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=128,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=0, max_leaves=32,
             min_child_weight=10.0, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=2000, n_jobs=-1,
             num_parallel_tree=None, ...)

In [ ]:
iso_global, cal_lang = evaluate_model(
    mdl,
    Xw_tr, y_tr_w, path_tr, zrms_tr, lang_tr,
    Xw_te, y_te_w, path_te, zrms_te, lang_te
)


=== TRAIN (clip, raw) ===  MAE=0.2818 RMSE=0.3745 CCC=0.8934
=== TEST (raw) ===  MAE=0.4852  RMSE=0.6100  CCC=0.6495
=== TEST (calibrado GLOBAL) ===  MAE=0.4707  RMSE=0.6024  CCC=0.7063
=== TEST (calibrado POR-IDIOMA) ===  MAE=0.4595  RMSE=0.5998  CCC=0.7127


lo siguiente se ha incorporado en la nueva funcion "evaluate_model"

In [ ]:
ytc_cls = np.digitize(ytc_te, bins=[2.5, 3.5], right=True)
ypc_cls = np.digitize(apply_iso_per_lang(ypc_te, langs_clip_te, cal_lang), bins=[2.5, 3.5], right=True)
print("\nBalanced Acc (H/M/L):", balanced_accuracy_score(ytc_cls, ypc_cls))
print(classification_report(ytc_cls, ypc_cls,
      target_names=["Bajo","Medio","Alto"], digits=4))


Balanced Acc (H/M/L): 0.5952706336188182
              precision    recall  f1-score   support

        Bajo     0.7737    0.5215    0.6230       721
       Medio     0.5517    0.7057    0.6193      1050
        Alto     0.6127    0.5586    0.5844       657

    accuracy                         0.6112      2428
   macro avg     0.6460    0.5953    0.6089      2428
weighted avg     0.6341    0.6112    0.6110      2428



In [ ]:
agg_kwargs = {"trim":0.10,"alpha":0.0}

In [ ]:
save_model_bundle(
    mdl, iso_global, cal_lang,
    "/content/drive/MyDrive/TP1/modelos/valencia/v3/segundo_nuevo",
    target_name="valence",
    positive_lang="es",
    xgb_params=best_params,
    n_features=Xw_tr.shape[1]
)

Guardado en: /content/drive/MyDrive/TP1/modelos/valencia/v3/segundo_nuevo


## validacion por folds

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
import numpy as np

In [ ]:
best_params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "n_estimators": 2000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "max_leaves": 32,
    "min_child_weight": 10.0,
    "subsample": 0.75,
    "colsample_bytree": 0.80,
    "colsample_bynode": 0.70,
    "gamma": 0.20,
    "reg_alpha": 1.0,
    "reg_lambda": 35.0,
    "max_bin": 128,
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "n_jobs": -1,
    "early_stopping_rounds": 50,
    "eval_metric": "rmse"
}

In [ ]:
def concordance_ccc(y_true, y_pred):
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    std_true = np.std(y_true)
    std_pred = np.std(y_pred)
    return (2 * cov) / (std_true**2 + std_pred**2 + (mean_true - mean_pred)**2 + 1e-8)

In [ ]:
ccc_scores = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(Xw_tr):
    # Train/val splits con arrays
    X_tr, X_val = Xw_tr[train_idx], Xw_tr[val_idx]
    y_tr, y_val = y_tr_w[train_idx], y_tr_w[val_idx]
    w_tr = W_full[train_idx]

    path_tr_fold = path_tr[train_idx]
    path_val_fold = path_tr[val_idx]
    zrms_tr_fold = zrms_tr[train_idx]
    zrms_val_fold = zrms_tr[val_idx]
    lang_tr_fold = lang_tr[train_idx]
    lang_val_fold = lang_tr[val_idx]

    # --- Modelo base ---
    model = XGBRegressor(**best_params)
    model.fit(X_tr, y_tr, sample_weight=w_tr,
              eval_set=[(X_val, y_val)],
              verbose=150)

    # --- Predicciones entrenamiento (para calibrar) ---
    yp_tr = model.predict(X_tr).astype(np.float32)
    yhat_tr_clip = aggregate_clip(path_tr_fold, yp_tr, zrms=zrms_tr_fold.ravel(), trim=0.10, alpha=0.0)

    ytrue_tr_clip = {}
    for p, yt in zip(path_tr_fold, y_tr):
        g = clip_group_from_path(p)
        ytrue_tr_clip.setdefault(g, []).append(float(yt))
    ytrue_tr_clip = {g: float(np.mean(v)) for g, v in ytrue_tr_clip.items()}

    keys_tr = sorted(set(yhat_tr_clip) & set(ytrue_tr_clip))
    ypc_tr = np.array([yhat_tr_clip[k] for k in keys_tr], np.float32)
    ytc_tr = np.array([ytrue_tr_clip[k] for k in keys_tr], np.float32)
    clip2lang_tr = lang_by_clip_from_paths(path_tr_fold, lang_tr_fold)
    langs_clip_tr = np.array([clip2lang_tr[k] for k in keys_tr])

    # --- Calibrador por idioma ---
    calibradores = fit_iso_per_lang(ypc_tr, ytc_tr, langs_clip_tr)

    # --- Predicciones validación ---
    yp_val = model.predict(X_val).astype(np.float32)
    yhat_val_clip = aggregate_clip(path_val_fold, yp_val, zrms=zrms_val_fold.ravel(), trim=0.10, alpha=0.0)

    ytrue_val_clip = {}
    for p, yt in zip(path_val_fold, y_val):
        g = clip_group_from_path(p)
        ytrue_val_clip.setdefault(g, []).append(float(yt))
    ytrue_val_clip = {g: float(np.mean(v)) for g, v in ytrue_val_clip.items()}

    keys_val = sorted(set(yhat_val_clip) & set(ytrue_val_clip))
    ypc_val = np.array([yhat_val_clip[k] for k in keys_val], np.float32)
    ytc_val = np.array([ytrue_val_clip[k] for k in keys_val], np.float32)
    clip2lang_val = lang_by_clip_from_paths(path_val_fold, lang_val_fold)
    langs_clip_val = np.array([clip2lang_val[k] for k in keys_val])

    # --- Calibrar predicciones ---
    ypc_val_cal = np.array([
        calibradores[lang].predict([[yp]])[0] if lang in calibradores else yp
        for yp, lang in zip(ypc_val, langs_clip_val)
    ])

    # --- CCC calibrado ---
    ccc = concordance_ccc(ytc_val, ypc_val_cal)
    ccc_scores.append(ccc)
    print("CCC: ", ccc)

print(f"\n=== CCC por fold calibrado === Mean= {np.mean(ccc_scores):.4f}, Std= {np.std(ccc_scores):.4f}")

[0]	validation_0-rmse:0.91815
[150]	validation_0-rmse:0.66022
[300]	validation_0-rmse:0.63212
[450]	validation_0-rmse:0.61706
[600]	validation_0-rmse:0.60653
[750]	validation_0-rmse:0.59865
[900]	validation_0-rmse:0.59194
[1050]	validation_0-rmse:0.58638
[1200]	validation_0-rmse:0.58164
[1350]	validation_0-rmse:0.57766
[1500]	validation_0-rmse:0.57378
[1650]	validation_0-rmse:0.57025
[1800]	validation_0-rmse:0.56738
[1950]	validation_0-rmse:0.56457
[1999]	validation_0-rmse:0.56368
CCC:  0.7976202
[0]	validation_0-rmse:0.91902
[150]	validation_0-rmse:0.65939
[300]	validation_0-rmse:0.63156
[450]	validation_0-rmse:0.61631
[600]	validation_0-rmse:0.60584
[750]	validation_0-rmse:0.59784
[900]	validation_0-rmse:0.59127
[1050]	validation_0-rmse:0.58626
[1200]	validation_0-rmse:0.58177
[1350]	validation_0-rmse:0.57773
[1500]	validation_0-rmse:0.57445
[1650]	validation_0-rmse:0.57126
[1800]	validation_0-rmse:0.56813
[1950]	validation_0-rmse:0.56515
[1999]	validation_0-rmse:0.56421
CCC:  0.7934